[Reference](https://blog.gopenai.com/langgraph-vs-crewai-vs-google-adk-i-built-the-same-agent-three-times-87c2f2ce3b59?sk=925f3939186761e9ea4c47978a53d6f1$0)

# LangGraph: Explicit Graph, Explicit Control


In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class State(TypedDict):
    query: str
    research: str
    final_answer: str
def research_node(state: State) -> State:
    # Call your research agent/tool here
    state["research"] = f"Research findings for: {state['query']}"
    return state
def writer_node(state: State) -> State:
    # Call your writer agent, using state["research"]
    state["final_answer"] = f"Final answer using: {state['research']}"
    return state
graph = StateGraph(State)
graph.add_node("research", research_node)
graph.add_node("writer", writer_node)
graph.set_entry_point("research")
graph.add_edge("research", "writer")
graph.add_edge("writer", END)
app = graph.compile()
result = app.invoke({"query": "What is agentic AI?"})

# CrewAI: Roles and Goals, Framework Handles the Rest


In [2]:
from crewai import Agent, Task, Crew

researcher = Agent(
    role="Research Specialist",
    goal="Find accurate information on the given topic",
    backstory="An expert at gathering and synthesizing information."
)
writer = Agent(
    role="Content Writer",
    goal="Write a clear final answer using research findings",
    backstory="An expert at turning research into clear prose."
)
research_task = Task(
    description="Research: {query}",
    agent=researcher,
    expected_output="A summary of relevant findings"
)
writing_task = Task(
    description="Write a final answer using the research",
    agent=writer,
    expected_output="A clear, well-written final answer",
    context=[research_task]
)
crew = Crew(agents=[researcher, writer], tasks=[research_task, writing_task])
result = crew.kickoff(inputs={"query": "What is agentic AI?"})

# Google ADK: Hierarchical, Production-Minded From the Start

In [3]:
from google.adk.agents import Agent
from google.adk.runners import Runner

research_agent = Agent(
    name="researcher",
    model="gemini-2.0-flash",
    instruction="Research the given topic thoroughly and return findings.",
)
writer_agent = Agent(
    name="writer",
    model="gemini-2.0-flash",
    instruction="Write a clear final answer using the research findings provided.",
    sub_agents=[research_agent]  # hierarchical relationship
)
runner = Runner(agent=writer_agent, app_name="research_writer_app")
result = runner.run(user_id="user1", session_id="session1",
                     new_message="What is agentic AI?")